# ATML PA2 - Task 3: GRPO

**Before running:** Settings → Accelerator **GPU T4** (x1 or x2), Internet **On**.

Rough T4 time: ~1-1.5 h. `outputs/task3_grpo/standard` is the GRPO policy needed later in Task 4 - keep the zip.

## 1. Setup

In [ ]:
# Fresh session: clone; re-run: pull the latest code.
%cd /kaggle/working
!if [ -d pa2 ]; then git -C pa2 pull; else git clone -b pa2 https://github.com/rana-taqveem/atml_internal.git pa2; fi
%cd /kaggle/working/pa2
!pip install -q -r requirements.txt
!python -m scripts.download_assets
!python -m scripts.validate_assets

In [ ]:
!python -m scripts.check_environment   # expect CUDA available: True, GPU: Tesla T4

## 2. Smoke test (~5 min)

In [ ]:
!python -m task3_grpo.continue_train --config configs/grpo.yaml --run-name smoke --output outputs/task3_grpo/smoke --updates 2
!python -m task3_grpo.evaluate --config configs/grpo.yaml --adapter outputs/task3_grpo/smoke --name smoke --n-prompts 8
!head -30 results/task3_grpo/smoke/train_summary.json

In [ ]:
!rm -rf outputs/task3_grpo/smoke results/task3_grpo/smoke

## 3. Step 1 - standard 20-update continuation + held-out evaluation

In [ ]:
!python -m task3_grpo.continue_train --config configs/grpo.yaml --run-name standard 2>&1 | tee -a task3.log
!python -m task3_grpo.evaluate --config configs/grpo.yaml --adapter checkpoints/grpo_midpoint_policy --name midpoint 2>&1 | tee -a task3.log
!python -m task3_grpo.evaluate --config configs/grpo.yaml --adapter none --name sft 2>&1 | tee -a task3.log
!python -m task3_grpo.evaluate --config configs/grpo.yaml --adapter outputs/task3_grpo/standard --name standard 2>&1 | tee -a task3.log

## 4. Step 2 - equal-generation group-size study (CPU, seconds)

In [ ]:
!python -m task3_grpo.analyze_group_size --config configs/grpo.yaml 2>&1 | tee -a task3.log

## 5. Step 3 - canonical GRPO vs Dr. GRPO forks

In [ ]:
!python -m task3_grpo.compare_normalization --config configs/grpo.yaml --skip-existing 2>&1 | tee -a task3.log

## 6. Summary, figures, qualitative candidates

In [ ]:
!python -m task3_grpo.summarize --config configs/grpo.yaml 2>&1 | tee -a task3.log

## 7. Save (download `task3_outputs.zip` from the Output panel)

In [ ]:
!zip -qr /kaggle/working/task3_outputs.zip outputs/task3_grpo results/task3_grpo task3.log
!ls -lh /kaggle/working/task3_outputs.zip